# GeoAI Aquaculture Pond Identification Challenge — End-to-End Pipeline

This notebook builds a full pipeline for the FAO/ITU "GeoAI Aquaculture Pond
Identification Challenge" (Zindi): binary classification (`aquaculture pond` = 1,
`other` = 0) from 12 monthly Sentinel-1 (SAR) + Sentinel-2 (optical) composites
per location, with intentional monthly gaps and a train/test **temporal** shift.

**Scoring:** `0.6 * F1 + 0.4 * ROC-AUC` on the leaderboard, with a rubric-based
trustworthiness review for the top 5 — so this notebook also produces
feature-importance / explainability output alongside the model.

**Before running:** place `Train.csv`, `Test.csv`, and `SampleSubmission.csv`
in the `DATA_DIR` set below (defaults to the notebook's own folder).

**Important — column names:** the challenge PDF describes the *bands* present
(S1: VH, VV; S2: blue, green, red, red-edge 1/2/3, NIR, narrow NIR, SWIR 1/2)
organised as 12 monthly composites, but the exact column-naming convention in
the real `Train.csv` isn't know to me yet (only the challenge description PDF
was available, not the CSVs). **Section 2** inspects the real columns and
Section 3 has a single configurable parser — adjust the regex patterns there
once you see your actual column names, and everything downstream adapts
automatically.


## 1. Setup

In [ ]:
# Uncomment/run once if these aren't installed in your environment
# %pip install lightgbm xgboost catboost shap scikit-learn pandas numpy matplotlib --quiet


In [ ]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score, roc_auc_score, precision_recall_curve, classification_report
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Optional but recommended libraries — code below degrades gracefully if missing
try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False
    print("lightgbm not installed — run: pip install lightgbm")

try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False
    print("shap not installed (optional, for explainability) — run: pip install shap")


In [ ]:
DATA_DIR = Path(".")   # <-- change if your CSVs live elsewhere
TRAIN_PATH = DATA_DIR / "Train.csv"
TEST_PATH = DATA_DIR / "Test.csv"
SAMPLE_SUB_PATH = DATA_DIR / "SampleSubmission.csv"
OUTPUT_DIR = Path("./outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

print("Train shape:", train_raw.shape)
print("Test shape:", test_raw.shape)
print("Sample submission shape:", sample_sub.shape)


## 2. Inspect the real data

Run this before touching Section 3. Look at:
- the ID column name
- the target column name and its unique values
- the full list of feature columns, so you can see the actual band/month
  naming convention (e.g. `VH_m1`, `m1_VH`, `VH_2024_01`, etc.)


In [ ]:
display(train_raw.head())
print()
print("Columns:")
for c in train_raw.columns:
    print(" -", c)


In [ ]:
print("Sample submission columns:", list(sample_sub.columns))
sample_sub.head()


In [ ]:
# Quick look at how many -9999 sentinel values are present, and target balance
n_sentinel = (train_raw.select_dtypes(include=[np.number]) == -9999).sum().sum()
print("Total -9999 cells in train (numeric cols):", n_sentinel)

# Adjust TARGET_COL / ID_COL once you've confirmed the real names from Section 2 output
TARGET_COL_GUESS = [c for c in train_raw.columns if c.lower() in ("target", "label", "class", "pond")]
ID_COL_GUESS = [c for c in train_raw.columns if c.lower() in ("id", "location_id", "uid")]
print("Guessed target column(s):", TARGET_COL_GUESS)
print("Guessed ID column(s):", ID_COL_GUESS)


## 3. Configuration — set these from what you saw in Section 2

Fill these in based on the real column names. `BAND_PATTERNS` maps a canonical
band name to a regex that matches that band's 12 monthly columns, with a
capture group for the month index. Two common conventions are provided as
starting points — edit them to match your file.


In [ ]:
# --- REQUIRED: set these to match your actual Train.csv / Test.csv ---
ID_COL = ID_COL_GUESS[0] if ID_COL_GUESS else "ID"
TARGET_COL = TARGET_COL_GUESS[0] if TARGET_COL_GUESS else "Target"

# Canonical band names we expect, per the challenge description:
# Sentinel-1 (SAR): VH, VV
# Sentinel-2 (optical): blue, green, red, red_edge1, red_edge2, red_edge3,
#                        nir, nir_narrow, swir1, swir2
BAND_ALIASES = {
    "VH": ["VH", "vh"],
    "VV": ["VV", "vv"],
    "blue": ["blue", "B2", "B02"],
    "green": ["green", "B3", "B03"],
    "red": ["red", "B4", "B04"],
    "red_edge1": ["rededge1", "red_edge1", "B5", "B05"],
    "red_edge2": ["rededge2", "red_edge2", "B6", "B06"],
    "red_edge3": ["rededge3", "red_edge3", "B7", "B07"],
    "nir": ["nir", "B8", "B08"],
    "nir_narrow": ["nirnarrow", "nir_narrow", "narrownir", "B8A"],
    "swir1": ["swir1", "B11"],
    "swir2": ["swir2", "B12"],
}

SENTINEL_MISSING_VALUE = -9999

def build_month_band_regex(band_key_variants):
    """Try a few common naming conventions: band_m1, m1_band, band_1, 1_band."""
    variants = "|".join(re.escape(v) for v in band_key_variants)
    patterns = [
        re.compile(rf"^(?:{variants})[_\-]?m?(\d{{1,2}})$", re.IGNORECASE),
        re.compile(rf"^m?(\d{{1,2}})[_\-]?(?:{variants})$", re.IGNORECASE),
        re.compile(rf"^(?:{variants})[_\-]?month[_\-]?(\d{{1,2}})$", re.IGNORECASE),
        re.compile(rf"^month[_\-]?(\d{{1,2}})[_\-]?(?:{variants})$", re.IGNORECASE),
    ]
    return patterns

def map_band_columns(columns):
    """Returns {canonical_band: {month_int: column_name}}."""
    mapping = {band: {} for band in BAND_ALIASES}
    unmatched = []
    for col in columns:
        if col in (ID_COL, TARGET_COL):
            continue
        matched = False
        for band, aliases in BAND_ALIASES.items():
            for pattern in build_month_band_regex(aliases):
                m = pattern.match(col)
                if m:
                    month = int(m.group(1))
                    mapping[band][month] = col
                    matched = True
                    break
            if matched:
                break
        if not matched:
            unmatched.append(col)
    return mapping, unmatched

band_month_map, unmatched_cols = map_band_columns(train_raw.columns)
print("Columns matched per band (should be up to 12 each):")
for band, months in band_month_map.items():
    print(f"  {band}: {len(months)} months matched")
print()
print(f"Unmatched columns ({len(unmatched_cols)}):", unmatched_cols[:30])


**If the auto-detected match counts above are 0 or wrong for most bands**,
the naming convention differs from what's assumed here — open `unmatched_cols`,
find the real pattern, and edit `build_month_band_regex` (or just hand-write
`band_month_map` directly as a dict of dicts) before moving on. Everything
from Section 4 onward only depends on `band_month_map`, `ID_COL`, and
`TARGET_COL`, so once this cell reports sensible matches you're set.


## 4. Recode sentinel missing values and reshape to long format

In [ ]:
def to_long(df, band_month_map, id_col):
    """Wide (one row per location) -> long (one row per location-month)."""
    all_months = sorted({m for months in band_month_map.values() for m in months})
    records = []
    ids = df[id_col].values
    for band, months in band_month_map.items():
        for month, col in months.items():
            vals = df[col].values.astype(float)
            vals = np.where(vals == SENTINEL_MISSING_VALUE, np.nan, vals)
            records.append(pd.DataFrame({id_col: ids, "month": month, "band": band, "value": vals}))
    long_df = pd.concat(records, ignore_index=True)
    long_pivot = long_df.pivot_table(index=[id_col, "month"], columns="band", values="value", aggfunc="first")
    long_pivot = long_pivot.reset_index()
    return long_pivot

train_long = to_long(train_raw, band_month_map, ID_COL)
test_long = to_long(test_raw, band_month_map, ID_COL)

print(train_long.shape, test_long.shape)
train_long.head()


## 5. Per-month spectral / radar index engineering

Water indices from the literature on Sentinel-1/2 aquaculture pond mapping:
- **NDWI** = (green − NIR) / (green + NIR)
- **MNDWI** = (green − SWIR1) / (green + SWIR1) — more robust to built-up/bare
  surface false positives than NDWI
- **NDVI** = (NIR − red) / (NIR + red) — helps exclude vegetated pixels
- SAR: **VH−VV**, **VH/VV**, and the **geometric mean of VH,VV** (in linear
  power, if your VH/VV are already in dB you may want to convert first —
  check your data's units in Section 2)


In [ ]:
def add_indices(long_df):
    df = long_df.copy()
    eps = 1e-6

    def safe_div(a, b):
        return (a - b) / (a + b + eps)

    if {"green", "nir"}.issubset(df.columns):
        df["NDWI"] = safe_div(df["green"], df["nir"])
    if {"green", "swir1"}.issubset(df.columns):
        df["MNDWI"] = safe_div(df["green"], df["swir1"])
    if {"nir", "red"}.issubset(df.columns):
        df["NDVI"] = safe_div(df["nir"], df["red"])
    if {"VH", "VV"}.issubset(df.columns):
        df["VH_minus_VV"] = df["VH"] - df["VV"]
        df["VH_over_VV"] = df["VH"] / (df["VV"].replace(0, np.nan))
        # geometric-mean-style combined backscatter (works directly if VH/VV are in dB,
        # since dB values are already log-domain — this is then just their average)
        df["VH_VV_combo"] = (df["VH"] + df["VV"]) / 2.0
    if {"NDWI", "NDVI"}.issubset(df.columns):
        df["NDWI_minus_NDVI"] = df["NDWI"] - df["NDVI"]
    return df

train_long = add_indices(train_long)
test_long = add_indices(test_long)
train_long.head()


## 6. Aggregate to one row per location

Deliberately **not** using raw per-month values as features (train/test come
from different time periods, so a "month 3" feature risks encoding season
rather than land cover). Instead: robust temporal statistics computed only
over valid (non-missing) months, plus missingness/valid-window features.

- **median** is prioritised over mean (more robust to outliers, better
  preserves narrow structures like pond dikes in the literature)
- **std/IQR** capture temporal *stability* — permanent ponds should look
  stable across months; seasonal water (paddies, flooding) should not
- separate valid-month counts per sensor, since S1 and S2 drop out for
  different reasons (S1 rarely missing; S2 missing under cloud cover)


In [ ]:
FEATURE_BASE_COLS = [c for c in train_long.columns if c not in (ID_COL, "month")]

def aggregate_features(long_df, id_col):
    agg_funcs = ["median", "mean", "std", "min", "max"]
    grouped = long_df.groupby(id_col)[FEATURE_BASE_COLS].agg(agg_funcs)
    grouped.columns = [f"{col}_{stat}" for col, stat in grouped.columns]

    # IQR per column
    q75 = long_df.groupby(id_col)[FEATURE_BASE_COLS].quantile(0.75)
    q25 = long_df.groupby(id_col)[FEATURE_BASE_COLS].quantile(0.25)
    iqr = (q75 - q25).add_suffix("_iqr")

    grouped = grouped.join(iqr)

    # missingness / valid-window features
    valid_counts = pd.DataFrame(index=grouped.index)
    for band in ["VH", "VV"]:
        if band in long_df.columns:
            valid_counts[f"{band}_valid_months"] = long_df.groupby(id_col)[band].apply(lambda s: s.notna().sum())
    optical_bands = [b for b in ["blue", "green", "red", "nir", "swir1", "swir2"] if b in long_df.columns]
    if optical_bands:
        valid_counts["S2_valid_months"] = long_df.groupby(id_col)[optical_bands].apply(
            lambda d: d.notna().any(axis=1).sum()
        )
    if {"VH", "green"}.issubset(long_df.columns):
        valid_counts["both_sensors_valid_months"] = long_df.groupby(id_col).apply(
            lambda d: (d["VH"].notna() & d["green"].notna()).sum()
        )
    valid_counts["total_months_present"] = long_df.groupby(id_col)["month"].nunique()

    grouped = grouped.join(valid_counts)
    grouped = grouped.reset_index()
    return grouped

train_features = aggregate_features(train_long, ID_COL)
test_features = aggregate_features(test_long, ID_COL)

print(train_features.shape, test_features.shape)
train_features.head()


In [ ]:
# Attach the target back onto the aggregated train features
train_features = train_features.merge(train_raw[[ID_COL, TARGET_COL]], on=ID_COL, how="left")
print(train_features[TARGET_COL].value_counts(normalize=True))


## 7. Adversarial validation

Train a classifier to distinguish train rows from test rows using only the
engineered features. A high AUC here confirms a real train/test distribution
shift (expected, since they're from different time periods) and tells us
*which* features are driving it — those are candidates to downweight or to
use as a stratification variable for CV, since plain random CV would be
overly optimistic if it ignores this shift.


In [ ]:
feature_cols = [c for c in train_features.columns if c not in (ID_COL, TARGET_COL)]

adv_train = train_features[feature_cols].copy()
adv_test = test_features[feature_cols].copy()
adv_X = pd.concat([adv_train, adv_test], ignore_index=True)
adv_y = np.array([0] * len(adv_train) + [1] * len(adv_test))  # 1 = test

if HAS_LGB:
    adv_model = lgb.LGBMClassifier(n_estimators=200, max_depth=4, random_state=RANDOM_STATE, verbosity=-1)
else:
    adv_model = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=RANDOM_STATE)

skf_adv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
adv_oof = np.zeros(len(adv_X))
for tr_idx, val_idx in skf_adv.split(adv_X, adv_y):
    adv_model.fit(adv_X.iloc[tr_idx].fillna(-999), adv_y[tr_idx])
    adv_oof[val_idx] = adv_model.predict_proba(adv_X.iloc[val_idx].fillna(-999))[:, 1]

adv_auc = roc_auc_score(adv_y, adv_oof)
print(f"Adversarial validation AUC (train vs test separability): {adv_auc:.3f}")
print("  ~0.5 = no detectable shift | closer to 1.0 = strong shift (expected here)")

adv_model.fit(adv_X.fillna(-999), adv_y)
if HAS_LGB:
    importances = pd.Series(adv_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
else:
    importances = pd.Series(adv_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("\nTop features separating train from test (biggest shift drivers):")
print(importances.head(15))


**How to use the result above:** if the adversarial AUC is high (say > 0.75)
and dominated by a handful of features, consider dropping those specific
features from the model or, better, keeping them but making sure your CV
splits are stratified on a discretized version of the top shift-driver (e.g.
bin `total_months_present` and stratify on it) so local validation reflects
the real difficulty rather than an inflated score.


## 8. Cross-validation, modeling, and F1 threshold tuning

Strategy: use Stratified K-Fold on the target (and optionally also stratify
on the top adversarial-shift feature, binned, using `StratifiedKFold` on a
combined key) — modelled with three complementary learners:

1. **LightGBM** (handles missing values natively, strong default choice for
   tabular satellite features at this sample size)
2. **Random Forest** (different bias, robust with the imputed feature set,
   good baseline for temporal generalization per the literature)
3. **Logistic Regression** on a scaled/imputed reduced set (interpretable
   anchor model)

We get out-of-fold (OOF) predicted probabilities for each, blend them, then
tune a single decision threshold on the blended OOF probabilities to
maximize F1 — since F1 is 60% of the leaderboard score and is threshold
sensitive, while AUC (the other 40%) only cares about ranking and needs no
threshold.


In [ ]:
X = train_features[feature_cols].copy()
y = train_features[TARGET_COL].astype(int).values
X_test = test_features[feature_cols].copy()

N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

oof_lgb = np.zeros(len(X))
oof_rf = np.zeros(len(X))
oof_lr = np.zeros(len(X))
test_pred_lgb = np.zeros(len(X_test))
test_pred_rf = np.zeros(len(X_test))
test_pred_lr = np.zeros(len(X_test))

lgb_models, rf_models, lr_models = [], [], []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]

    # --- LightGBM (native NaN handling) ---
    if HAS_LGB:
        lgb_model = lgb.LGBMClassifier(
            n_estimators=500,
            learning_rate=0.03,
            num_leaves=15,
            max_depth=5,
            min_child_samples=15,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=RANDOM_STATE + fold,
            verbosity=-1,
        )
        lgb_model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            eval_metric="auc",
            callbacks=[lgb.early_stopping(50, verbose=False)],
        )
        oof_lgb[val_idx] = lgb_model.predict_proba(X_val)[:, 1]
        test_pred_lgb += lgb_model.predict_proba(X_test)[:, 1] / N_SPLITS
        lgb_models.append(lgb_model)

    # --- Random Forest (needs imputed input) ---
    imputer = SimpleImputer(strategy="median")
    X_tr_imp = imputer.fit_transform(X_tr)
    X_val_imp = imputer.transform(X_val)
    X_test_imp = imputer.transform(X_test)

    rf_model = RandomForestClassifier(
        n_estimators=600, max_depth=8, min_samples_leaf=3,
        class_weight="balanced_subsample", random_state=RANDOM_STATE + fold, n_jobs=-1,
    )
    rf_model.fit(X_tr_imp, y_tr)
    oof_rf[val_idx] = rf_model.predict_proba(X_val_imp)[:, 1]
    test_pred_rf += rf_model.predict_proba(X_test_imp)[:, 1] / N_SPLITS
    rf_models.append((rf_model, imputer))

    # --- Logistic Regression (scaled + imputed, interpretable anchor) ---
    lr_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
        ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", C=0.5, random_state=RANDOM_STATE)),
    ])
    lr_pipe.fit(X_tr, y_tr)
    oof_lr[val_idx] = lr_pipe.predict_proba(X_val)[:, 1]
    test_pred_lr += lr_pipe.predict_proba(X_test)[:, 1] / N_SPLITS
    lr_models.append(lr_pipe)

    fold_auc = roc_auc_score(y_val, oof_lgb[val_idx] if HAS_LGB else oof_rf[val_idx])
    print(f"Fold {fold+1}: AUC = {fold_auc:.4f}")


In [ ]:
def best_f1_threshold(y_true, y_prob):
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_prob)
    f1s = 2 * precisions * recalls / (precisions + recalls + 1e-9)
    best_idx = np.nanargmax(f1s[:-1])  # last point has no corresponding threshold
    return thresholds[best_idx], f1s[best_idx]

# Individual model OOF scores
for name, oof in [("LightGBM", oof_lgb), ("RandomForest", oof_rf), ("LogisticRegression", oof_lr)]:
    if name == "LightGBM" and not HAS_LGB:
        continue
    auc = roc_auc_score(y, oof)
    thr, f1 = best_f1_threshold(y, oof)
    weighted = 0.6 * f1 + 0.4 * auc
    print(f"{name:20s} AUC={auc:.4f}  best-thr={thr:.3f}  F1={f1:.4f}  weighted_score={weighted:.4f}")


In [ ]:
# Blend (simple average of OOF probabilities across available models)
components = [oof_lgb] if HAS_LGB else []
components += [oof_rf, oof_lr]
oof_blend = np.mean(components, axis=0)

test_components = [test_pred_lgb] if HAS_LGB else []
test_components += [test_pred_rf, test_pred_lr]
test_blend = np.mean(test_components, axis=0)

blend_auc = roc_auc_score(y, oof_blend)
blend_thr, blend_f1 = best_f1_threshold(y, oof_blend)
blend_weighted = 0.6 * blend_f1 + 0.4 * blend_auc

print(f"Blended OOF  AUC={blend_auc:.4f}  best-thr={blend_thr:.3f}  F1={blend_f1:.4f}  weighted_score={blend_weighted:.4f}")
print()
print(classification_report(y, (oof_blend >= blend_thr).astype(int)))


**Note on the test-set class-shift warning:** the organizers flag that the
test set may have a higher proportion of positives than the ~40% seen in
train. The threshold above is tuned purely on train-distribution OOF data —
if you want to hedge against that shift, it's reasonable to nudge the
threshold slightly lower (biasing toward predicting more positives) and
compare a couple of candidate thresholds using the reasoning in Section 7's
shift diagnostics, since there's no ground truth to directly validate this
adjustment against.


## 9. Feature importance / explainability (for the trustworthiness rubric)

In [ ]:
if HAS_LGB:
    importances = pd.DataFrame({
        "feature": feature_cols,
        "importance": np.mean([m.feature_importances_ for m in lgb_models], axis=0),
    }).sort_values("importance", ascending=False)

    plt.figure(figsize=(8, 8))
    top_n = importances.head(20)
    plt.barh(top_n["feature"][::-1], top_n["importance"][::-1])
    plt.title("Top 20 LightGBM feature importances (avg across folds)")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "feature_importance.png", dpi=150)
    plt.show()

    display(importances.head(20))
else:
    print("Install lightgbm to get gain-based feature importances here.")


In [ ]:
if HAS_SHAP and HAS_LGB:
    explainer = shap.TreeExplainer(lgb_models[0])
    shap_values = explainer.shap_values(X)
    shap.summary_plot(shap_values, X, max_display=20, show=False)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "shap_summary.png", dpi=150)
    plt.show()
else:
    print("Install shap (pip install shap) for per-prediction explanations, e.g. for write-up in the rubric phase.")


## 10. Build the submission file

In [ ]:
final_predictions = (test_blend >= blend_thr).astype(int)

submission = test_features[[ID_COL]].copy()

# Match SampleSubmission.csv's column names/order exactly
sub_cols = list(sample_sub.columns)
if len(sub_cols) >= 2:
    id_col_sub, target_col_sub = sub_cols[0], sub_cols[1]
else:
    id_col_sub, target_col_sub = ID_COL, TARGET_COL

submission = submission.rename(columns={ID_COL: id_col_sub})
submission[target_col_sub] = final_predictions
# If the competition wants probabilities instead of hard labels, use:
# submission[target_col_sub] = test_blend

# Sanity-check against sample submission
assert set(submission[id_col_sub]) == set(sample_sub[id_col_sub]), "ID mismatch vs SampleSubmission.csv!"
submission = submission.set_index(id_col_sub).loc[sample_sub[id_col_sub]].reset_index()

submission_path = OUTPUT_DIR / "submission.csv"
submission.to_csv(submission_path, index=False)
print("Saved:", submission_path)
submission.head()


## Summary / next steps

- If Section 3's auto-detected column matches look wrong, fix `band_month_map`
  first — everything else depends on it.
- Watch the **adversarial validation AUC** (Section 7) as your primary signal
  for how serious the train/test shift is, and revisit which features are
  driving it if your leaderboard score diverges a lot from local CV.
- Try swapping in **XGBoost** or **CatBoost** as additional blend members if
  installed — the same OOF/threshold-tuning pattern in Section 8 applies
  directly.
- For the rubric/trustworthiness phase, pair the SHAP output (Section 9) with
  a short written explanation of the physical meaning of your top features
  (e.g. "low, stable MNDWI/VH across available months" = permanent water),
  and be explicit about the limitations noted earlier: no lat/lon for spatial
  validation, and an unavoidable train/test temporal gap.
